# tRNA41 Dependency Map

Predict the secondary structure of tRNA-Ala (tRNA41, chr1) using three different
approaches, then compare each against the known cloverleaf contact map.

**Sections:**
1. Setup & ground truth
2. **RiNALMo epistasis geometry** — double-mutant embeddings, `epi_R_singles` metric
3. **SpeciesLM log-odds dependency map** — MLM head, single-pass O(N)
4. **SpeciesLM embedding perturbation map** — cosine distance on token embeddings, single-pass O(N)
5. Summary comparison

## 1. Setup & ground truth

In [ ]:
import sys
from pathlib import Path

# Ensure genebeddings is importable (editable install or sys.path fallback)
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations, product
from scipy.stats import pearsonr
from seqmat import SeqMat

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_logodds_dependency_map,
    compute_embedding_perturbation_map,
)

In [ ]:
# --- Model config (change these to switch models) ---
from genebeddings.wrappers import RiNALMoWrapper, SpeciesLMWrapper

# Epistasis metrics model
EPI_MODEL_CLS = RiNALMoWrapper
EPI_CONTEXT = 511
EPI_DB_PATH = 'embeddings/rinalmo.db'

# Dependency map model (needs MLM head for log-odds; any model works for embedding perturbation)
DEPMAP_MODEL_CLS = SpeciesLMWrapper
DEPMAP_CONTEXT = 600

# --- tRNA41 (tRNA-Ala) coordinates ---
CHROM = 'chr1'
START = 159141611
END = 159141684
GENE = 'tRNA41'
CHROM_NUM = '1'
STRAND = 'N'  # negative strand
MAX_DISTANCE = 100
BASES = 'ATGC'
PAD = 256  # flanking context for dependency maps

### Helpers

In [ ]:
BASES = 'ATGC'

def get_all_epistasis_ids(seq, indices, gene, chrom, zyg="N", max_distance=100):
    """Generate epistasis IDs for all position pairs within max_distance, all non-ref ALT combos."""
    items = []
    for i, j in combinations(range(len(seq)), 2):
        if j - i > max_distance:
            continue
        pos1, pos2 = int(indices[i]), int(indices[j])
        ref1, ref2 = seq[i], seq[j]
        if ref1 not in BASES or ref2 not in BASES:
            continue
        alts1 = [b for b in BASES if b != ref1]
        alts2 = [b for b in BASES if b != ref2]
        for alt1, alt2 in product(alts1, alts2):
            id1 = f"{gene}:{chrom}:{pos1}:{ref1}:{alt1}:{zyg}"
            id2 = f"{gene}:{chrom}:{pos2}:{ref2}:{alt2}:{zyg}"
            items.append({'epistasis_id': f"{id1}|{id2}", 'pos1': pos1, 'pos2': pos2})
    return pd.DataFrame(items)


def dotbracket_to_contact_map(ss):
    """Convert dot-bracket notation to a symmetric binary contact matrix."""
    opens = "([{<ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    closes = ")]}>abcdefghijklmnopqrstuvwxyz"
    N = len(ss)
    M = np.zeros((N, N), dtype=int)
    stack = {ch: [] for ch in opens}
    for i, ch in enumerate(ss):
        if ch in opens:
            stack[ch].append(i)
        elif ch in closes:
            j = stack[opens[closes.index(ch)]].pop()
            M[i, j] = M[j, i] = 1
    return M


def upper_tri_flatten(M):
    i, j = np.triu_indices_from(M, k=1)
    return M[i, j]


def zero_diagonal(M):
    """Zero out the diagonal of a matrix."""
    M = M.copy()
    np.fill_diagonal(M, 0)
    return M


def correlate_with_structure(pred_matrix, M_true, label, flip=True):
    """Score a predicted map against the true contact map.
    
    Zeros out the diagonal of both maps before comparing (upper triangle Pearson r).
    """
    pred = pred_matrix.copy()
    if flip:
        pred = pred[::-1, ::-1]
    pred[np.isnan(pred)] = 0
    pred = zero_diagonal(pred)
    truth = zero_diagonal(M_true)
    true_flat = upper_tri_flatten(truth)
    pred_flat = upper_tri_flatten(pred)
    r, p = pearsonr(true_flat, pred_flat)
    print(f"{label}: Pearson r = {r:.4f}, p = {p:.2e}")
    return r, p


def plot_heatmap(matrix, title, cmap='magma', figsize=(10, 8)):
    """Plot a symmetric heatmap."""
    plt.figure(figsize=figsize)
    sns.heatmap(matrix, cmap=cmap, robust=True)
    plt.title(title)
    plt.xlabel("Position")
    plt.ylabel("Position")
    plt.tight_layout()
    plt.show()

### Ground truth: tRNA secondary structure

In [ ]:
# tRNA41 cloverleaf (reverse-complement strand, gaps stripped from alignment)
ss_raw = '.>>>>>>>..>>>>..........<<<<.>>>>>.........................<<<<<................>>>>>.......<<<<<<<<<<<<.'
ss_raw = ss_raw.replace('>', '(').replace('<', ')')
refseq_raw = '.GTCTCTGTGGCGCAATGGAcgA.GCGCGCTGGACTTCTA..................ATCCAGAG...........GtTCCGGGTTCGAGTCCCGGCAGAGATG'
ss = ''.join(c for c, r in zip(ss_raw, refseq_raw) if r != '.')
M_true = dotbracket_to_contact_map(ss)

plt.figure(figsize=(6, 6))
plt.imshow(M_true, cmap='viridis', origin='upper')
plt.title("Ground truth: tRNA41 secondary structure")
plt.xlabel("Position")
plt.ylabel("Position")
plt.colorbar()
plt.show()
print(f"Structure: {len(ss)} nt, {M_true.sum() // 2} base pairs")

# Collect results for final comparison
all_results = []

## 2. RiNALMo: Epistasis geometry (double-mutant embeddings)

For every pair of positions (i, j), embed all 9 double-mutant combinations,
compute epistasis metrics (non-additivity of embedding effects), and
aggregate into a contact-like heatmap.

**Cost:** O(N^2) embeddings &mdash; slow but captures non-linear interactions.

In [ ]:
# Generate variant pairs
s = SeqMat.from_fasta('hg38', CHROM, START, END)
df = get_all_epistasis_ids(s.seq, s.index, GENE, CHROM_NUM, STRAND, MAX_DISTANCE)
print(f"{len(df)} epistasis pairs")

# Run RiNALMo
epi_model = EPI_MODEL_CLS()
db = VariantEmbeddingDB(EPI_DB_PATH)

epi_data = add_epistasis_metrics(
    df, db,
    model=epi_model,
    id_col="epistasis_id",
    context=EPI_CONTEXT,
    show_progress=True,
    force=False,
    batch_size=8,
)

# Plot: epi_R_singles
epi_name = EPI_MODEL_CLS.__name__.replace("Wrapper", "")
dep_singles = epi_data.pivot_table(index='pos1', columns='pos2', values='epi_R_singles', aggfunc='max')
dep_singles = dep_singles.combine_first(dep_singles.T)
plot_heatmap(dep_singles.iloc[::-1, ::-1], f"{epi_name}: epi_R_singles (epistasis geometry)")

# Score
r, p = correlate_with_structure(dep_singles.fillna(0).values, M_true, f"{epi_name} epistasis (epi_R_singles)", flip=False)
all_results.append((f"{epi_name} epistasis (epi_R_singles)", r, p))

del epi_model

## 3. SpeciesLM: Log-odds dependency map (MLM head)

Mutate position i, measure how nucleotide probabilities change at position j
via the masked language model head.

**Cost:** O(N) forward passes &mdash; fast, but requires an MLM model.

In [ ]:
depmap_model = DEPMAP_MODEL_CLS()
depmap_name = DEPMAP_MODEL_CLS.__name__.replace("Wrapper", "")
seq_padded = SeqMat.from_fasta('hg38', CHROM, START - PAD, END + PAD).seq
tRNA_positions = list(range(PAD, PAD + END - START + 1))

result_logodds = compute_logodds_dependency_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    show_progress=True,
)
result_logodds.plot(title=f"{depmap_name}: log-odds dependency map")

r, p = correlate_with_structure(result_logodds.matrix, M_true, f"{depmap_name} log-odds")
all_results.append((f"{depmap_name} log-odds", r, p))

## 4. SpeciesLM: Embedding perturbation map (cosine distance)

Mutate position i, re-embed, measure cosine distance at each token position j.

**Cost:** O(N) forward passes &mdash; fast, works with any model (no MLM head needed).

In [ ]:
result_embed = compute_embedding_perturbation_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    diff="cosine",
    show_progress=True,
)
result_embed.plot(title=f"{depmap_name}: embedding perturbation map (cosine)")

r, p = correlate_with_structure(result_embed.matrix, M_true, f"{depmap_name} embedding perturbation (cosine)")
all_results.append((f"{depmap_name} embedding perturbation (cosine)", r, p))

del depmap_model

## 5. Summary comparison

In [ ]:
df_results = pd.DataFrame(all_results, columns=['Method', 'Pearson r', 'p-value'])
df_results = df_results.sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_results.index += 1
df_results.index.name = 'Rank'
display(df_results)

# Bar chart
fig, ax = plt.subplots(figsize=(10, max(3, len(df_results) * 0.6)))
colors = ['C2' if 'epistasis' in m else 'C1' if 'log-odds' in m else 'C0' for m in df_results['Method']]
ax.barh(range(len(df_results)), df_results['Pearson r'], color=colors)
ax.set_yticks(range(len(df_results)))
ax.set_yticklabels(df_results['Method'], fontsize=10)
ax.set_xlabel('Pearson r (vs tRNA secondary structure)')
ax.set_title('tRNA41 Contact Prediction: Method Comparison')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='C0', label='Embedding perturbation'),
    Patch(color='C1', label='Log-odds'),
    Patch(color='C2', label='Epistasis geometry'),
], loc='lower right')
plt.tight_layout()
plt.show()